In [41]:
# Importing Libraries

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV

In [42]:
cleaned_customer_transaction_data = pd.read_csv('/Users/Dell/Frudulent-transaction-detection-for-Finlora-Company/Finlora_Dataset/Artifacts/EDA_Data.csv')

Threshold-based features were created based on key behavioural and risk indicators identified during exploratory analysis. These include low KYC verification levels, location mismatches between the registered country and IP address, and high-risk transaction characteristics such as high transaction amounts, elevated IP risk scores, and low device trust scores. These features were identified as high-risk thresholds due to their significantly elevated fraud rates. These thresholds were converted into binary risk indicators to enhance the model's ability to detect suspicious transactions.

In [43]:
# Creating a threshold based features from the following risk_signal

from pyexpat import features


cleaned_customer_transaction_data['timestamp'] = pd.to_datetime(cleaned_customer_transaction_data['timestamp'])
cleaned_customer_transaction_data['late_night_hours'] = ((cleaned_customer_transaction_data['hour'] >= 3) & cleaned_customer_transaction_data['hour'] <= 7).astype(int)
cleaned_customer_transaction_data['amount_high'] = (cleaned_customer_transaction_data['amount_usd'] >= 1000).astype(int)
cleaned_customer_transaction_data['high_ip_risk'] = (cleaned_customer_transaction_data['ip_risk_score'] > 0.8).astype(int)
cleaned_customer_transaction_data['low_device_trust'] = (cleaned_customer_transaction_data['device_trust_score'] < 0.5).astype(int)
cleaned_customer_transaction_data['new_account'] = ((cleaned_customer_transaction_data['account_age_days'] >= 30) & (cleaned_customer_transaction_data['account_age_days'] < 90)).astype(int)
cleaned_customer_transaction_data['very_new_account'] = (cleaned_customer_transaction_data['account_age_days'] < 30).astype(int)
cleaned_customer_transaction_data['velocity_spike'] = (cleaned_customer_transaction_data['txn_velocity_1h'] >=3 ).astype(int)

high_risk_signal_features = cleaned_customer_transaction_data[['late_night_hours', 'amount_high', 'high_ip_risk', 'low_device_trust', 'new_account', 'very_new_account', 'velocity_spike']] 
high_risk_signal_features.head()

,late_night_hours,amount_high,high_ip_risk,low_device_trust,new_account,very_new_account,velocity_spike
0,1,0,0,0,0,0,0
1,1,0,0,1,0,0,0
2,1,0,0,0,0,0,0
3,1,0,0,0,0,0,0
4,1,0,0,0,0,0,0


In [52]:
# Checking the features in the dataset
list(cleaned_customer_transaction_data.columns)

['timestamp',
 'home_country',
 'source_currency',
 'dest_currency',
 'channel',
 'amount_src',
 'amount_usd',
 'fee',
 'new_device',
 'ip_country',
 'location_mismatch',
 'ip_risk_score',
 'kyc_tier',
 'account_age_days',
 'device_trust_score',
 'risk_score_internal',
 'txn_velocity_1h',
 'txn_velocity_24h',
 'corridor_risk',
 'is_fraud',
 'hour',
 'day_of_week',
 'is_weekend',
 'late_night_hours',
 'amount_high',
 'high_ip_risk',
 'low_device_trust',
 'new_account',
 'very_new_account',
 'velocity_spike']

### Feature Selection

In [45]:
# Drop some of the features which are not required for model building

# Dropping all temporary buckets columns
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(['account_age_bucket', 'device_trust_score_bucket', 'ip_risk_score_bucket', 'amount_usd_bucket'], axis=1)

# Dropping all identifiers columns
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(['transaction_id', 'customer_id', 'device_id', 'ip_address'], axis=1)

# Dropping irrelevant variables
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(['chargeback_history_count', 'month', 'exchange_rate_src_to_dest'], axis=1)

### Defining categorical features

In [46]:
categorical_features = cleaned_customer_transaction_data.select_dtypes(include=['object', 'bool']).columns 

categorical_features

Index(['home_country', 'source_currency', 'dest_currency', 'channel',
       'new_device', 'ip_country', 'location_mismatch', 'kyc_tier'],
      dtype='object')

### Defining numerical features

In [47]:
numerical_features = cleaned_customer_transaction_data.select_dtypes(include=['int', 'float']).columns.drop('is_fraud')

numerical_features

Index(['amount_src', 'amount_usd', 'fee', 'ip_risk_score', 'account_age_days',
       'device_trust_score', 'risk_score_internal', 'txn_velocity_1h',
       'txn_velocity_24h', 'corridor_risk', 'hour', 'day_of_week',
       'is_weekend', 'late_night_hours', 'amount_high', 'high_ip_risk',
       'low_device_trust', 'new_account', 'very_new_account',
       'velocity_spike'],
      dtype='object')

In [48]:
print(f"Categorical: {len(categorical_features)}")
print(f"Numerical: {len(numerical_features)}")
print(f"dataset: {cleaned_customer_transaction_data.shape}")

Categorical: 8
Numerical: 20
dataset: (10780, 30)


In [49]:
cleaned_customer_transaction_data.columns

Index(['timestamp', 'home_country', 'source_currency', 'dest_currency',
       'channel', 'amount_src', 'amount_usd', 'fee', 'new_device',
       'ip_country', 'location_mismatch', 'ip_risk_score', 'kyc_tier',
       'account_age_days', 'device_trust_score', 'risk_score_internal',
       'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud',
       'hour', 'day_of_week', 'is_weekend', 'late_night_hours', 'amount_high',
       'high_ip_risk', 'low_device_trust', 'new_account', 'very_new_account',
       'velocity_spike'],
      dtype='object')

In [50]:
cleaned_customer_transaction_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10780 entries, 0 to 10779
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   timestamp            10780 non-null  datetime64[ns, UTC]
 1   home_country         10756 non-null  object             
 2   source_currency      10780 non-null  object             
 3   dest_currency        10780 non-null  object             
 4   channel              10780 non-null  object             
 5   amount_src           10780 non-null  float64            
 6   amount_usd           10780 non-null  float64            
 7   fee                  10780 non-null  float64            
 8   new_device           10780 non-null  bool               
 9   ip_country           10752 non-null  object             
 10  location_mismatch    10780 non-null  bool               
 11  ip_risk_score        10780 non-null  float64            
 12  kyc_tier          

In [51]:
cleaned_customer_transaction_data.to_csv("../Finlora_Dataset/Artifacts/Engineered_Data.csv", index=False)